# Ensemble Candidate Comparison

Dummy Classifier, Decision Tree, Random Forest를 동일한 validation에서 비교합니다.
현재 공개 범위는 2-1 실습의 문제 1-1까지이며, 미완료 실습은 포함하지 않았습니다.

## Experiment Rules

- 악성 종양을 positive class 1로 변환합니다.
- train / validation / test를 60% / 20% / 20%로 분리합니다.
- 모델 선택에는 validation AP를 사용합니다.
- test는 최종 선택 전까지 사용하지 않습니다.

In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, f1_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

SEED = 42

data = load_breast_cancer(as_frame=True)
X = data.data
y = (data.target == 0).astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=SEED
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED
)

print("shape:", X.shape)
print("split:", len(X_train), len(X_valid), len(X_test))
print("positive=malignant:", round(float(y.mean()), 4))

assert X.shape == (569, 30)
assert set(y.unique()) == {0, 1}
assert set(X_train.index).isdisjoint(X_valid.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_valid.index).isdisjoint(X_test.index)

shape: (569, 30)
split: 341 114 114
positive=malignant: 0.3726


In [2]:
def compare_ensemble_candidates(models, X_train, y_train, X_valid, y_valid):
    rows = []
    fitted_models = {}

    for name, model in models.items():
        fitted = model.fit(X_train, y_train)
        probability = fitted.predict_proba(X_valid)[:, 1]
        prediction = probability >= 0.5

        rows.append({
            "model": name,
            "AP": average_precision_score(y_valid, probability),
            "F1": f1_score(y_valid, prediction),
            "Recall": recall_score(y_valid, prediction),
        })
        fitted_models[name] = fitted

    table = pd.DataFrame(rows).sort_values("AP", ascending=False).reset_index(drop=True)
    selected_name = str(table.iloc[0]["model"])
    return table, selected_name, fitted_models

In [3]:
models = {
    "dummy": DummyClassifier(strategy="prior"),
    "tree": DecisionTreeClassifier(random_state=SEED),
    "forest": RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        class_weight="balanced",
        max_features="sqrt",
    ),
}

table, selected_name, fitted_models = compare_ensemble_candidates(
    models, X_train, y_train, X_valid, y_valid
)
selected_template = fitted_models[selected_name]

print(table.round(4).to_string(index=False))
print("selected:", selected_name)

assert table["model"].nunique() == 3
assert table[["AP", "F1", "Recall"]].apply(
    lambda column: column.between(0.0, 1.0).all()
).all()
assert selected_name == str(table.iloc[0]["model"])

 model     AP     F1  Recall
forest 0.9933 0.9524  0.9302
  tree 0.8834 0.9157  0.8837
 dummy 0.3772 0.0000  0.0000
selected: forest


## Interpretation

이 분할에서는 Random Forest의 validation AP가 가장 높았습니다. Dummy 기준선은 복잡한 모델이 실제로 단순 예측보다 나은지 확인하기 위해 함께 비교했습니다. 이 결과는 하나의 고정 분할에서 얻었으므로 모든 데이터에 일반화할 수 없습니다.

> 이 실습은 학습 목적이며 실제 의료 판단에 사용할 수 없습니다.